# IndoBERT fine-tuning — BALANCED 3000 (label Opus + rubrik domain-aware)

Fine-tune `indobenchmark/indobert-base-p1` (3 kelas) pada subset **balanced 1000/kelas**
dengan label TERBARU (re-label Claude Opus + rubrik domain-aware, 2026-07-01).
**Self-contained** (gaya `indobert_finetune_colab_variant.ipynb`): baca koleksi
**`processed_bert`** dari MongoDB Atlas lewat flag **`in_balanced3k=True`** — tanpa upload
CSV / clone repo, cukup `MONGO_URI`. Split kanonik identik SVM (urut comment_id, seed=42,
70/20/10 -> 2100/600/300).

**Sebelum mulai:** Runtime -> Change runtime type -> **GPU (T4)**. Lalu Run all.
Tempel `MONGO_URI` saat diminta (atau simpan sbg Colab Secret `MONGO_URI`).

## 0. Dependency + cek GPU

In [ ]:
%pip install -q "transformers>=4.40" "torch" "pymongo[srv]" dnspython certifi scikit-learn matplotlib pandas numpy
import torch
print("CUDA tersedia:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU saja (lambat!)")

## 1. Baca `processed_bert` (flag `in_balanced3k`) + split kanonik

Tarik teks `bert` + `label` untuk baris ber-flag `in_balanced3k=True` (label sudah versi
domain-aware terbaru), peta ke `label_id`, lalu split 70/20/10 IDENTIK SVM: urut
`comment_id` -> test 10% -> val (`0.20/0.90` sisa) -> train, `stratify` + `seed=42`.
Tempel `MONGO_URI` (butuh IP allowlist `0.0.0.0/0` agar Colab bisa konek).

In [ ]:
import os, pandas as pd
from pymongo import MongoClient
import certifi
from sklearn.model_selection import train_test_split

FLAG = "in_balanced3k"   # penanda subset balanced 1000/kelas (label domain-aware)

MONGO_URI = os.environ.get("MONGO_URI", "")
if not MONGO_URI:
    try:
        from google.colab import userdata
        MONGO_URI = userdata.get("MONGO_URI")
    except Exception:
        MONGO_URI = ""
if not MONGO_URI:
    from getpass import getpass
    MONGO_URI = getpass("MONGO_URI: ")
DB_NAME = os.environ.get("MONGO_DB_NAME", "youtube_sentiment")
client = MongoClient(MONGO_URI, tlsCAFile=certifi.where(), serverSelectionTimeoutMS=20000)
client.admin.command("ping"); print("Koneksi MongoDB OK.")

LABELS = ["Negatif", "Netral", "Positif"]; LABEL2ID = {l: i for i, l in enumerate(LABELS)}
df = pd.DataFrame(list(client[DB_NAME]["processed_bert"].find(
    {FLAG: True}, {"_id": 0, "comment_id": 1, "bert": 1, "label": 1})))
assert len(df), f"Tak ada dokumen ber-flag {FLAG}=True. Jalankan src.modeling.push_subset_ids dulu."
df["label_id"] = df["label"].map(LABEL2ID)
df = df.sort_values("comment_id").reset_index(drop=True)
tmp, df_test = train_test_split(df, test_size=0.10, stratify=df["label_id"], random_state=42)
df_train, df_val = train_test_split(tmp, test_size=0.20/0.90, stratify=tmp["label_id"], random_state=42)
print(f"n={len(df)} | train={len(df_train)} val={len(df_val)} test={len(df_test)}")
print(df["label"].value_counts().to_string())

## 2. Tokenizer + Dataset (subword WordPiece, MAX_LEN=128)

In [ ]:
from transformers import AutoTokenizer
import torch

MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LEN = 128
SEED = 42
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

class DS(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tok(list(texts), truncation=True, max_length=MAX_LEN, padding=True)
        self.labels = list(labels)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i]); return item

ds_train = DS(df_train["bert"].astype(str), df_train["label_id"])
ds_val   = DS(df_val["bert"].astype(str),   df_val["label_id"])
ds_test  = DS(df_test["bert"].astype(str),  df_test["label_id"])
print("Dataset siap.")

## 3. Model + TrainingArguments (4 epoch, best-val checkpoint)

In [ ]:
import numpy as np
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, EarlyStoppingCallback, set_seed)
from sklearn.metrics import f1_score, accuracy_score

set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3,
    id2label={i: l for i, l in enumerate(LABELS)}, label2id=LABEL2ID)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"macro_f1": f1_score(p.label_ids, preds, average="macro"),
            "accuracy": accuracy_score(p.label_ids, preds)}

MAX_EPOCHS, PATIENCE = 4, 3
_kw = dict(output_dir="indobert_out", num_train_epochs=MAX_EPOCHS,
           per_device_train_batch_size=16, per_device_eval_batch_size=32,
           learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1,
           save_total_limit=2, load_best_model_at_end=True,
           metric_for_best_model="macro_f1", greater_is_better=True,
           seed=SEED, logging_steps=50, report_to="none")
try:
    args = TrainingArguments(eval_strategy="epoch", save_strategy="epoch", **_kw)
except TypeError:
    args = TrainingArguments(evaluation_strategy="epoch", save_strategy="epoch", **_kw)

trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                  compute_metrics=compute_metrics,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
print("Trainer siap.")

## 4. Fine-tune

In [ ]:
trainer.train()
print("Selesai. Val terbaik:", trainer.evaluate())

## 5. Evaluasi test set (metrik setara SVM)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pred = trainer.predict(ds_test)
y_pred = np.argmax(pred.predictions, axis=1).tolist()
y_true = df_test["label_id"].tolist()

def evaluate(y_true, y_pred, labels=LABELS):
    ids = list(range(len(labels)))
    rep = classification_report(y_true, y_pred, labels=ids, target_names=labels, output_dict=True, zero_division=0)
    return {"accuracy": round(accuracy_score(y_true, y_pred), 4),
            "macro_f1": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4),
            "weighted_f1": round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4),
            "per_class": {l: {k: round(rep[l][k], 4) for k in ["precision", "recall", "f1-score"]} | {"support": int(rep[l]["support"])} for l in labels},
            "confusion_matrix": confusion_matrix(y_true, y_pred, labels=ids).tolist(), "labels": labels}

m_test = evaluate(y_true, y_pred)
print("="*60); print("  IndoBERT — TEST (balanced3k)"); print("="*60)
print(f"  Accuracy : {m_test['accuracy']:.4f}")
print(f"  Macro-F1 : {m_test['macro_f1']:.4f}")
for l in LABELS:
    pc = m_test["per_class"][l]
    print(f"    {l:<10} P={pc['precision']:.3f} R={pc['recall']:.3f} F1={pc['f1-score']:.3f}")

## 6. Confusion matrix + simpan metrik

Simpan `indobert_balanced3k_metrics.json` (struktur dibaca `compare_models --tag balanced3k`).
**Unduh JSON itu** dan letakkan di `outputs/reports/` repo lokal (atau kirim ke Claude).

In [ ]:
import matplotlib.pyplot as plt, json
cm = np.array(m_test["confusion_matrix"])
fig, ax = plt.subplots(figsize=(5, 4.3))
im = ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(3), LABELS); ax.set_yticks(range(3), LABELS)
ax.set_xlabel("Prediksi"); ax.set_ylabel("Aktual"); ax.set_title(f"IndoBERT — Test (acc={m_test['accuracy']:.3f})")
th = cm.max()/2
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > th else "black")
fig.colorbar(im, ax=ax, fraction=0.046); fig.tight_layout(); plt.show()

json.dump({"model": "IndoBERT", "dataset": "balanced3k", "test": m_test},
          open("indobert_balanced3k_metrics.json", "w"), ensure_ascii=False, indent=2)
fig.savefig("indobert_balanced3k_test_confusion.png", dpi=120)
print("Tersimpan: indobert_balanced3k_metrics.json + indobert_balanced3k_test_confusion.png")
try:
    from google.colab import files
    files.download("indobert_balanced3k_metrics.json")
    files.download("indobert_balanced3k_test_confusion.png")
except Exception as ex:
    print("Download manual dari panel Files. Detail:", ex)